# BirdCLEF 2026 — Autonomous Research Agent

## Structure
- **Setup (runs once):** Load data, encode labels, split train/val, build data generator
- **Agent loop (repeats):** LLM proposes model → train → evaluate → feed results back → iterate

## Imports

In [1]:
import pandas as pd
import numpy as np
import librosa
import ast
import re
import time
import tensorflow as tf
import ollama
from sklearn.model_selection import train_test_split
from experiment_log import new_run_id, add_experiment, print_summary
import experiment_log

print(tf.config.list_physical_devices())

import importlib
import prompt_builder
importlib.reload(prompt_builder)
from prompt_builder import build_prompt
importlib.reload(experiment_log)
from experiment_log import new_run_id, add_experiment, print_summary


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


---
## SETUP — Runs once before the agent loop

---
### Step 1: Load and filter `train.csv`

- Keep only rated recordings (`rating > 0`) — removes ~37% of unreviewed recordings
- Keep only the columns we need: `filename`, `primary_label`, `secondary_labels`
- Build the full filepath so Python can find each audio file on disk

> **Note:** BASE_PATH must point to the data folder on your machine.
> Update it if you move the project.

In [2]:
BASE_PATH = "/Users/pedrofernandes/Desktop/APA-7th_1/"

df = pd.read_csv(BASE_PATH + "data/train.csv")
df = df[df["rating"] > 0].reset_index(drop=True)
df = df[["filename", "primary_label", "secondary_labels"]]
df["filepath"] = BASE_PATH + "data/train_audio/" + df["filename"]

print(f"Recordings after filtering: {len(df)}")
print(df.head(3))

Recordings after filtering: 22700
               filename primary_label secondary_labels  \
0  1595929/XC999239.ogg       1595929               []   
1    22930/XC940773.ogg         22930               []   
2    22956/XC900618.ogg         22956               []   

                                            filepath  
0  /Users/pedrofernandes/Desktop/APA-7th_1/data/t...  
1  /Users/pedrofernandes/Desktop/APA-7th_1/data/t...  
2  /Users/pedrofernandes/Desktop/APA-7th_1/data/t...  


---
### Step 2: Explore the data

Basic exploration before training — understanding class distribution,
imbalance, and data quality. This informs what the agent will be working with.

In [3]:
full_df = pd.read_csv(BASE_PATH + "data/train.csv")

print("=== Dataset overview ===")
print(f"Total recordings         : {len(full_df)}")
print(f"After rating filter      : {len(df)}")
print(f"Removed (rating == 0)    : {len(full_df) - len(df)}")
print()

print("=== Class breakdown (after filter) ===")
print(df.merge(full_df[["filename","class_name"]], on="filename")["class_name"].value_counts())
print()

print("=== Species recording counts ===")
counts = df["primary_label"].value_counts()
print(f"Total unique species in training : {len(counts)}")
print(f"Species with 1 recording         : {(counts == 1).sum()}")
print(f"Species with < 10 recordings     : {(counts < 10).sum()}")
print(f"Most common species              : {counts.index[0]} ({counts.iloc[0]} recordings)")
print(f"Least common species             : {counts.index[-1]} ({counts.iloc[-1]} recordings)")

=== Dataset overview ===
Total recordings         : 35549
After rating filter      : 22700
Removed (rating == 0)    : 12849

=== Class breakdown (after filter) ===
class_name
Aves        22609
Amphibia       58
Mammalia       33
Name: count, dtype: int64

=== Species recording counts ===
Total unique species in training : 185
Species with 1 recording         : 9
Species with < 10 recordings     : 24
Most common species              : yeofly1 (415 recordings)
Least common species             : 74580 (1 recordings)


---
### Step 3: Label encoding

- Encoding is based on `taxonomy.csv` (234 species), NOT on what appears in `train.csv`
- This ensures the submission columns are always in the correct order
- 28 species exist in taxonomy but have no training audio — they get no training examples
- We keep `secondary_labels` because this is a **multi-label** problem:
  multiple species can be present in a single recording simultaneously

In [4]:
taxonomy = pd.read_csv(BASE_PATH + "data/taxonomy.csv")
NUM_CLASSES = len(taxonomy)  # 234

label_to_idx = {label: idx for idx, label in enumerate(taxonomy["primary_label"])}

print(f"Total species (submission columns): {NUM_CLASSES}")
print(f"Species in training data         : {df['primary_label'].nunique()}")
print(f"Species with no training data    : {NUM_CLASSES - df['primary_label'].nunique()}")

Total species (submission columns): 234
Species in training data         : 185
Species with no training data    : 49


---
### Step 4: Train / Validation split

- **Stratified** split: each species is proportionally represented in both sets
- **Rare species** (< 2 recordings) cannot be stratified — kept entirely in training
- Split is **fixed once** — the same split is used across every agent iteration
  so scores are comparable between experiments
- 80% train / 20% validation

> Note: species with only 1 recording cannot be validated locally.
> This is a known limitation — acknowledged in the report Limitations section.

In [5]:
counts = df["primary_label"].value_counts()
rare_labels = counts[counts < 2].index

rare_df   = df[df["primary_label"].isin(rare_labels)]
common_df = df[~df["primary_label"].isin(rare_labels)]

train_df, val_df = train_test_split(
    common_df,
    test_size=0.2,
    random_state=42,
    stratify=common_df["primary_label"]
)

# rare species go entirely to training
train_df = pd.concat([train_df, rare_df]).reset_index(drop=True)

print(f"Total recordings : {len(df)}")
print(f"Train recordings : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val   recordings : {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Rare species kept entirely in train: {len(rare_labels)}")

Total recordings : 22700
Train recordings : 18161 (80.0%)
Val   recordings : 4539 (20.0%)
Rare species kept entirely in train: 9


---
### Step 5: Configuration

Mel-spectrogram parameters are defined here — **not hardcoded inside the generator**.
This allows the agent loop to change them between experiments.

We start with **low resolution** (`n_mels=64`) as recommended:
faster training means the agent can iterate more quickly.
Once the pipeline is stable, the agent can try higher resolutions.

In [6]:
# --- Audio config ---
SAMPLE_RATE = 32000
DURATION    = 5

# --- Mel-spectrogram config ---
N_MELS  = 64
F_MAX   = 16000

# --- Training config ---
BATCH_SIZE = 32
N_EPOCHS   = 3

# --- Agent config ---
LLM_MODEL    = "gemma4:e4b"
N_ITERATIONS = 1

# --- Backbone config ---
# Options: "efficientnet" | "mobilenet" | "resnet" | "scratch"
BACKBONE   = "efficientnet"
FINE_TUNE  = True   # if True, backbone weights are updated during training (lower LR used)

BACKBONE_CLASS_NAME = {
    "efficientnet": "EfficientNetB0",
    "mobilenet"   : "MobileNetV2",
    "resnet"      : "ResNet50",
    "scratch"     : "custom CNN",
}

# ── Build backbone ────────────────────────────────────────────────────────────

def build_backbone(name, input_shape=(64, 313, 1)):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Lambda(lambda t: tf.repeat(t, 3, axis=-1))(inputs)

    if name == "efficientnet":
        base = tf.keras.applications.EfficientNetB0(
            include_top=False, weights="imagenet", pooling="avg"
        )
        feature_dim = 1280
    elif name == "mobilenet":
        base = tf.keras.applications.MobileNetV2(
            include_top=False, weights="imagenet", pooling="avg"
        )
        feature_dim = 1280
    elif name == "resnet":
        base = tf.keras.applications.ResNet50(
            include_top=False, weights="imagenet", pooling="avg"
        )
        feature_dim = 2048
    elif name == "scratch":
        print("Backbone: scratch — LLM will define the full CNN architecture")
        return None, None
    else:
        raise ValueError(f"Unknown backbone: {name}")

    features = base(x)
    backbone_model = tf.keras.Model(inputs=inputs, outputs=features, name=f"backbone_{name}")
    base.trainable = FINE_TUNE
    print(f"Backbone: {BACKBONE_CLASS_NAME[name]} | feature_dim: {feature_dim} | weights: ImageNet | trainable: {FINE_TUNE}")
    return backbone_model, feature_dim

backbone_model, feature_dim = build_backbone(BACKBONE)

Backbone: EfficientNetB0 | feature_dim: 1280 | weights: ImageNet | trainable: True


---
### Step 6: Audio loading and mel-spectrogram conversion

Two helper functions:
- `load_audio`: loads an `.ogg` file, resamples to fixed rate, pads if shorter than 5 seconds
- `audio_to_melspectrogram`: converts raw audio into a 2D image (frequency × time)

In [7]:
def load_audio(filepath, sample_rate=SAMPLE_RATE, duration=DURATION):
    audio, sr = librosa.load(filepath, sr=sample_rate, duration=duration)

    # pad with silence if recording is shorter than target duration
    target_length = sample_rate * duration
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))

    return audio


def audio_to_melspectrogram(audio, sample_rate=SAMPLE_RATE, n_mels=N_MELS, fmax=F_MAX):
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_mels=n_mels,
        fmax=fmax
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)  # convert to log scale
    return mel_db

---
### Step 7: Multi-label encoding

Each recording can have multiple species present simultaneously.
The label is a binary vector of length 234:
- `1` at position `i` means species `i` is present
- `0` means absent

Example: `[0, 0, 1, 0, 1, 0, ...]` — species 2 and 4 are both present.

This is different from multi-class (which assumes exactly one species per recording).

In [8]:
def encode_labels(primary_label, secondary_labels_str, label_to_idx, num_classes):
    label_vector = np.zeros(num_classes, dtype=np.float32)

    # primary species is always present
    if primary_label in label_to_idx:
        label_vector[label_to_idx[primary_label]] = 1.0

    # secondary species — parse the string list and add them
    try:
        secondary = ast.literal_eval(secondary_labels_str)
        for sec in secondary:
            if sec in label_to_idx:
                label_vector[label_to_idx[sec]] = 1.0
    except (ValueError, SyntaxError):
        pass  # if parsing fails, only primary label is used

    return label_vector

---
### Step 8: Data generator

Loads audio and converts to spectrograms **on demand** during training —
never loads all 22,700 files into memory at once.

The `augment` flag is a placeholder for the agent to control.
When `augment=True`, the generator will apply random modifications to audio
before converting to spectrogram (the agent decides what augmentations to try).

In [9]:
class BirdDataGenerator(tf.keras.utils.Sequence):

    def __init__(self, dataframe, batch_size=BATCH_SIZE, augment=False, shuffle=True):
        self.df         = dataframe.reset_index(drop=True)
        self.batch_size = batch_size
        self.augment    = augment
        self.shuffle    = shuffle
        self.on_epoch_end()

    def __len__(self):
        return len(self.df) // self.batch_size

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size:(idx + 1) * self.batch_size]

        X, y = [], []

        for _, row in batch.iterrows():
            audio = load_audio(row["filepath"])

            # augmentation hook — agent-controlled, only applied during training
            if self.augment:
                audio = self._augment(audio)

            mel = audio_to_melspectrogram(audio)
            X.append(mel)

            label_vector = encode_labels(
                row["primary_label"],
                row["secondary_labels"],
                label_to_idx,
                NUM_CLASSES
            )
            y.append(label_vector)

        # add channel dimension: (batch, n_mels, time) → (batch, n_mels, time, 1)
        X = np.array(X)[..., np.newaxis]
        y = np.array(y)

        return X, y

    def _augment(self, audio):
        # placeholder — the LLM will fill this in with specific augmentation strategies
        # e.g. pitch shift, time stretch, add background noise
        return audio

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1).reset_index(drop=True)

---
### Step 9: Initialise generators and sanity check

- Training generator: shuffled, augmentation on
- Validation generator: not shuffled, no augmentation — must stay consistent across iterations

In [10]:
train_generator = BirdDataGenerator(train_df, batch_size=BATCH_SIZE, augment=True,  shuffle=True)
val_generator   = BirdDataGenerator(val_df,   batch_size=BATCH_SIZE, augment=False, shuffle=False)

# sanity check — load one batch and verify shapes
X_batch, y_batch = train_generator[0]

print(f"X shape: {X_batch.shape}")  # (32, 64, time_steps, 1)
print(f"y shape: {y_batch.shape}")  # (32, 234)
print(f"Species present in first sample : {np.where(y_batch[0] == 1)[0]}")
print(f"Train batches per epoch: {len(train_generator)}")
print(f"Val   batches per epoch: {len(val_generator)}")

X shape: (32, 64, 313, 1)
y shape: (32, 234)
Species present in first sample : [201]
Train batches per epoch: 567
Val   batches per epoch: 141


---
## AGENT LOOP — Starts here

Everything above this line runs **once**.
Everything below runs **once per iteration**.

---
### Step 10: Agent helpers

Three small functions:
- `call_llm` — sends the prompt to Ollama, returns the raw text response
- `extract_code` — pulls the Python code block out of the LLM response
- `execute_safely` — runs the code, catches any crash and returns the result

In [11]:
def call_llm(prompt, model=LLM_MODEL):
    response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
    return response["message"]["content"]


def extract_code(llm_response):
    match = re.search(r"```python\s*(.*?)```", llm_response, re.DOTALL)
    return match.group(1).strip() if match else None


def strip_forbidden_calls(code):
    # remove anything the LLM must not control
    forbidden = ["model.fit(", "train_generator", "val_generator",
                 "Conv1D(", "Conv2D(", "Conv3D(", "model.compile("]
    cleaned = []
    for line in code.splitlines():
        if any(term in line for term in forbidden):
            cleaned.append(f"# [removed]: {line}")
        else:
            cleaned.append(line)
    return "\n".join(cleaned)


def execute_safely(code):
    if code is None:
        return {"status": "crashed", "error": "No code block found",
                "val_auc": None, "val_loss": None, "training_time_sec": None, "epochs_trained": 0}

    if "backbone_model" not in code:
        return {"status": "crashed", "error": "LLM code does not use backbone_model",
                "val_auc": None, "val_loss": None, "training_time_sec": None, "epochs_trained": 0}

    try:
        real_train = train_generator
        real_val   = val_generator

        globals().pop("val_auc", None)
        globals().pop("val_loss", None)
        globals().pop("model",   None)

        clean_code = strip_forbidden_calls(code)

        start = time.time()
        exec(clean_code, globals())  # LLM builds the model architecture

        # lower LR when fine-tuning to avoid destroying pretrained weights
        lr = 1e-4 if FINE_TUNE else 1e-3
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
            loss="binary_crossentropy",
            metrics=[tf.keras.metrics.AUC(name="auc")]
        )

        globals()["train_generator"] = real_train
        globals()["val_generator"]   = real_val

        history = model.fit(train_generator, validation_data=val_generator, epochs=N_EPOCHS, verbose=0)
        elapsed = round(time.time() - start)

        train_loss = history.history["loss"][-1]
        val_auc    = history.history["val_auc"][-1]
        val_loss   = history.history["val_loss"][-1]
        print(f"train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_auc: {val_auc:.4f}")

        return {"status": "success", "error": None,
                "val_auc": val_auc, "val_loss": val_loss,
                "training_time_sec": elapsed, "epochs_trained": N_EPOCHS}

    except Exception as e:
        return {"status": "crashed", "error": str(e),
                "val_auc": None, "val_loss": None,
                "training_time_sec": None, "epochs_trained": 0}

---
### Step 11: Run the agent loop

In [12]:
run_id = new_run_id()

print(f"Starting agent run: {run_id}")
print(f"Model: {LLM_MODEL} | Backbone: {BACKBONE_CLASS_NAME[BACKBONE]} | Iterations: {N_ITERATIONS}")
print("-" * 50)

for iteration in range(1, N_ITERATIONS + 1):
    print(f"\n[Iteration {iteration}/{N_ITERATIONS}]")

    _feature_dim = int(backbone_model.output_shape[-1]) if backbone_model is not None else 1280
    prompt = build_prompt(_feature_dim, backbone_name=BACKBONE_CLASS_NAME[BACKBONE])

    print("  Calling LLM...")
    llm_response = call_llm(prompt)
    code = extract_code(llm_response)

    print("  Running generated code...")
    result = execute_safely(code)
    iteration_crash_count = 1 if result["status"] == "crashed" else 0

    if result["status"] == "crashed":
        print(f"  CRASHED: {result['error']}")
    else:
        print(f"  SUCCESS: val_auc={result['val_auc']} | val_loss={result['val_loss']}")

    add_experiment(
        run_id            = run_id,
        iteration         = iteration,
        label             = f"backbone_{BACKBONE}_iter{iteration}",
        architecture      = llm_response[:200],
        code              = code,
        status            = result["status"],
        crash_count       = iteration_crash_count,
        error             = result["error"],
        val_auc           = result["val_auc"],
        val_loss          = result["val_loss"],
        epochs_trained    = result["epochs_trained"],
        training_time_sec = result["training_time_sec"],
        llm_analysis      = "",
        notes             = f"backbone={BACKBONE}, fine_tune={FINE_TUNE}, epochs={N_EPOCHS}"
    )

print("\n" + "=" * 50)
print("Agent run complete.")
print_summary(run_id)

Starting agent run: run_20260516_111853
Model: gemma4:e4b | Backbone: EfficientNetB0 | Iterations: 1
--------------------------------------------------

[Iteration 1/1]
  Calling LLM...
  Running generated code...


/opt/anaconda3/envs/keras_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()
/opt/anaconda3/envs/keras_env/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


KeyboardInterrupt: 